# Macro Surprise + VWAP Execution : premier screening actions US

*Premi?re tentative pour tester si les journ?es de surprises macro?conomiques peuvent cr?er des patterns court terme sur une s?lection d'actions US.*

Ce notebook est le r?sum? lisible du projet. Les scripts Python restent le moteur du backtest. Ici, je charge seulement les CSV et les figures d?j? g?n?r?s, puis j'explique ce qui a ?t? test?, ce que les tests de robustesse veulent dire, et les limites ? garder en t?te.

## Id?e De Recherche

Je voulais tester si les surprises macro?conomiques peuvent aider ? rep?rer des opportunit?s court terme sur des actions individuelles.

L'id?e n'est pas simplement d'acheter les bonnes nouvelles macro et de vendre les mauvaises. Le but est de combiner un signal de surprise macro avec une r?gle d'ex?cution intraday plus disciplin?e autour du VWAP.

Pour cette version US, il faut ?tre pr?cis dans l'interpr?tation. Beaucoup de publications macro importantes sortent avant l'ouverture des actions US. Donc la version actuelle doit plut?t ?tre lue comme :

**jour de macro + VWAP intraday**

plut?t que :

**r?action pure minute par minute au timestamp de publication**

Cette nuance est importante. Si je montre ce travail ? un trader, je ne veux pas pr?tendre mesurer une r?action parfaite ? la seconde de publication alors que l'entr?e se fait souvent ? l'ouverture ou apr?s.

## Donn?es Utilis?es

L'?tude combine deux types de donn?es :

- publications macro?conomiques US, avec actual, estimate et previous quand disponible ;
- m?triques historiques de surprise par type d'?v?nement, utilis?es pour normaliser la surprise ;
- donn?es OHLCV minute sur les actions ;
- VWAP intraday calcul? ? partir des barres minute, avec reset chaque jour.

Pour ce premier screening, j'ai travaill? avec environ 10 ans de donn?es en timeframe 1 minute sur les actions test?es. Les fichiers minute sont lourds et je n'arrive pas ? les charger correctement ici, donc ils ne sont pas inclus dans le GitHub.

Si tu lis ce repo et que tu veux v?rifier les fichiers exacts utilis?s pour reproduire le test, contacte-moi et je peux te les envoyer par mail.

Les donn?es brutes ne sont pas incluses dans ce repo. Le notebook attend seulement des CSV de synth?se et des figures de rapport.


## Construction Du Signal

La surprise macro est construite en trois ?tapes :

```text
surprise_raw = actual - estimate
surprise_z = (surprise_raw - historical_average_surprise) / historical_surprise_std
signal = surprise_z * direction
```

`direction = +1` quand une valeur macro plus ?lev?e est consid?r?e comme positive.

`direction = -1` quand une valeur macro plus faible est consid?r?e comme positive.

Les ?v?nements marqu?s `MIXED` sont exclus parce que le signe n'est pas assez clair.

Le moteur teste ensuite deux hypoth?ses :

- `DRIFT` : trader dans le sens du signal macro.
- `FADE` : trader contre le signal macro.

## Logique D'ex?cution VWAP

La r?gle VWAP est utilis?e comme filtre d'ex?cution, pas comme signal macro en soi.

Pour un trade long, le mod?le pr?f?re entrer quand le prix est sous ou proche du VWAP. Pour un short, il pr?f?re entrer quand le prix est au-dessus ou proche du VWAP. Si le prix n'est pas favorable, le mod?le attend un retour vers le VWAP dans une fen?tre d?finie.

Deux modes d'ex?cution sont importants :

- `close_cross` : plus optimiste, car il suffit que la cl?ture de la bougie respecte la condition.
- `limit_touch` : plus conservateur, car il demande une logique de touch/fill plus proche d'un ordre limite autour du VWAP.

Le VWAP est remis ? z?ro chaque jour. C'est logique pour de l'ex?cution intraday, mais ?a veut aussi dire que la strat?gie peut devenir en partie un test de mean reversion autour du VWAP si je ne fais pas attention. C'est pour ?a que les tests random et train/test sont importants.

## Cadre De Robustesse

Le mod?le ne garde pas une action seulement parce que le backtest complet est profitable.

Les tests de robustesse sont :

- test avec sens de trade al?atoire ;
- test avec timestamp al?atoire ;
- placebo avec d?calage temporel ;
- sweep de co?ts de transaction ;
- split chronologique train/test ;
- walk-forward par ann?e.

Le split train/test est particuli?rement important. La r?gle est s?lectionn?e sur les premiers 70% de la p?riode, puis test?e sur les derniers 30%. Si la r?gle para?t bonne sur l'?chantillon complet mais perd de l'argent sur la p?riode de test, je la rejette ou je la consid?re comme fragile.

## Chargement Des Sorties

Le notebook attend les fichiers g?n?r?s dans :

```text
../reports/trader_report/csv_outputs/
```

Il cherche aussi les graphiques PNG dans :

```text
../reports/trader_report/figures/
```

Si un fichier manque, le notebook affiche un avertissement clair et continue.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image({self.filename})"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_DIR = PROJECT_ROOT / "reports" / "trader_report" / "csv_outputs"
FIG_DIR = PROJECT_ROOT / "reports" / "trader_report" / "figures"
NOTES_DIR = PROJECT_ROOT / "reports" / "trader_report" / "notes"

FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTES_DIR.mkdir(parents=True, exist_ok=True)

expected_files = {
    "master": "single_country_master_summary.csv",
    "all_configs": "single_country_all_configs_summary.csv",
    "random_tests": "single_country_random_tests.csv",
    "train_test": "single_country_train_test.csv",
    "walk_forward": "single_country_walk_forward.csv",
}

loaded = {}
missing = []

for key, filename in expected_files.items():
    path = CSV_DIR / filename
    if path.exists():
        loaded[key] = pd.read_csv(path)
        print(f"Charg? {filename} : {loaded[key].shape[0]} lignes, {loaded[key].shape[1]} colonnes")
    else:
        missing.append(filename)
        print(f"AVERTISSEMENT : fichier manquant {filename}")

print("\nDossier CSV :", CSV_DIR)
print("Dossier figures :", FIG_DIR)

if missing:
    display(Markdown("**Fichiers manquants :** " + ", ".join(f"`{m}`" for m in missing)))
else:
    display(Markdown("Tous les CSV attendus ont ?t? charg?s."))

master = loaded.get("master")
all_configs = loaded.get("all_configs")
random_tests = loaded.get("random_tests")
train_test = loaded.get("train_test")
walk_forward = loaded.get("walk_forward")

## Contr?les Sur Les Tickers

La pr?sentation des tickers peut sembler un d?tail, mais ?a peut cr?er de la confusion pendant une revue.

Je veux surtout rep?rer deux cas :

- un ticker affich? comme `LONDON-STRATEGIC-EDGE`, qui est probablement un probl?me d'extraction depuis le nom du fichier ;
- des tickers dupliqu?s, par exemple si le m?me CSV brut est pr?sent deux fois.

Le code ci-dessous ne modifie pas le r?sultat du backtest. Il signale seulement les probl?mes de pr?sentation.

In [ ]:
def extract_ticker_from_filename(name):
    base = Path(name).stem
    base = re.sub(r"\s*\(\d+\)$", "", base)
    m = re.match(r"([A-Za-z]+)_dataset", base)
    if m:
        return m.group(1).upper()
    first = re.split(r"[_\-\s]", base)[0]
    return first.upper() if first else base.upper()

notes = []

if master is not None and "ticker" in master.columns:
    tickers = master["ticker"].astype(str)
    bad = tickers.str.upper().eq("LONDON-STRATEGIC-EDGE")
    if bad.any():
        notes.append("`LONDON-STRATEGIC-EDGE` appara?t comme ticker dans le r?sum?. Je le traiterais comme un probl?me de pr?sentation, probablement caus? par l'extraction du nom de fichier. Si le fichier source correspond ? LMB, je l'afficherais comme LMB dans les tableaux sans changer le r?sultat brut.")

    dupes = tickers[tickers.duplicated(keep=False)].sort_values().unique().tolist()
    if dupes:
        notes.append("Tickers dupliqu?s dans le master summary : " + ", ".join(f"`{d}`" for d in dupes))

raw_dir = PROJECT_ROOT / "US-Actions"
if raw_dir.exists():
    raw_files = sorted(raw_dir.glob("*.csv"))
    raw_tickers = [extract_ticker_from_filename(p.name) for p in raw_files]
    raw_dupes = sorted({t for t in raw_tickers if raw_tickers.count(t) > 1})
    if raw_dupes:
        notes.append("Des fichiers bruts semblent dupliqu?s localement : " + ", ".join(f"`{d}`" for d in raw_dupes) + ". Je les nettoierais avant un run final.")
    if any(t == "LMB" for t in raw_tickers):
        notes.append("Un fichier source `LMB` existe localement. Si un r?sultat affiche `LONDON-STRATEGIC-EDGE`, je l'afficherais comme `LMB (?)` et j'ajouterais une note sur le probl?me de nom de fichier.")

if notes:
    display(Markdown("\n\n".join("- " + n for n in notes)))
else:
    display(Markdown("Aucun probl?me de ticker d?tect? dans les r?sum?s charg?s ou les noms de fichiers locaux."))

## Vue D'ensemble Des R?sultats

Cette section utilise `single_country_master_summary.csv`. Elle donne la vue globale : combien d'actions ont ?t? test?es, comment les classifications se r?partissent, quels noms ressortent le mieux par score de confiance, et quels noms sont rejet?s.

In [ ]:
def display_master_overview(master_df):
    if master_df is None:
        display(Markdown("`single_country_master_summary.csv` manque, donc la vue d'ensemble ne peut pas encore ?tre construite."))
        return None

    df = master_df.copy()

    if "ticker" in df.columns:
        bad = df["ticker"].astype(str).str.upper().eq("LONDON-STRATEGIC-EDGE")
        if bad.any():
            df.loc[bad, "ticker"] = "LMB (?)"

    n = len(df)
    display(Markdown(f"**Actions dans le master summary :** {n}"))

    if "classification" in df.columns:
        counts = df["classification"].fillna("MISSING").value_counts().rename_axis("classification").reset_index(name="count")
        display(counts)

    table_cols = [
        "ticker",
        "classification",
        "confidence_score",
        "best_hypothese",
        "best_vwap_mode",
        "best_horizon",
        "best_ret_moy_pct",
        "test_ret_moy_pct",
        "random_side_percentile",
        "random_ts_percentile",
        "cost_break_even_bps",
        "main_warning",
    ]
    present_cols = [c for c in table_cols if c in df.columns]

    sort_cols = [c for c in ["confidence_score", "test_ret_moy_pct", "best_ret_moy_pct"] if c in df.columns]
    ranked = df.sort_values(sort_cols, ascending=False) if sort_cols else df

    display(Markdown("**Meilleurs candidats par score de confiance**"))
    display(ranked[present_cols].head(12) if present_cols else ranked.head(12))

    if "classification" in df.columns:
        rejected_mask = df["classification"].astype(str).str.contains("D_REJECT|INSUFFICIENT", case=False, na=False)
        rejected = df.loc[rejected_mask]
        display(Markdown("**Noms rejet?s ou avec donn?es insuffisantes**"))
        if rejected.empty:
            display(Markdown("Aucune ligne rejet?e ou insuffisante dans le master summary charg?."))
        else:
            display(rejected[present_cols].head(30) if present_cols else rejected.head(30))

    return df

master_view = display_master_overview(master)

## Figures Existantes Ou Graphiques De Secours

Si des PNG existent d?j?, je les affiche directement. Sinon, le notebook essaie de g?n?rer quelques graphiques simples ? partir des CSV disponibles.

Les graphiques de secours restent volontairement sobres : titres lisibles, axes clairs, pas de style inutile.

In [ ]:
def save_current_fig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Sauvegard?", path)
    return path

existing_pngs = sorted(FIG_DIR.glob("*.png"))

if existing_pngs:
    display(Markdown("**Figures de rapport trouv?es :**"))
    for path in existing_pngs:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))
elif not HAS_MATPLOTLIB:
    display(Markdown("Aucune figure PNG trouv?e, et `matplotlib` n'est pas install? dans cet environnement. Installer les d?pendances permet de g?n?rer les graphiques de secours."))
else:
    display(Markdown("Aucune figure PNG trouv?e. Je g?n?re des graphiques de secours quand les donn?es le permettent."))

    generated = []

    if master_view is not None and "classification" in master_view.columns:
        counts = master_view["classification"].fillna("MISSING").value_counts()
        fig, ax = plt.subplots(figsize=(8, 4.5))
        counts.plot(kind="bar", ax=ax, color="#4c78a8")
        ax.set_title("Nombre d'actions par classification")
        ax.set_xlabel("Classification")
        ax.set_ylabel("Nombre d'actions")
        ax.tick_params(axis="x", rotation=35)
        generated.append(save_current_fig("01_classification_count.png"))

    if master_view is not None and {"ticker", "confidence_score"}.issubset(master_view.columns):
        top = master_view.sort_values("confidence_score", ascending=False).head(10).copy()
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top["ticker"].astype(str), top["confidence_score"], color="#59a14f")
        ax.invert_yaxis()
        ax.set_title("Top 10 des actions par score de confiance")
        ax.set_xlabel("Score de confiance")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("02_top_10_confidence_score.png"))

    if master_view is not None and {"ticker", "best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["best_ret_moy_pct", "test_ret_moy_pct"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["best_ret_moy_pct"], plot_df["test_ret_moy_pct"], color="#4c78a8", alpha=0.8)
            ax.axhline(0, color="#666666", linewidth=1)
            ax.axvline(0, color="#666666", linewidth=1)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["best_ret_moy_pct"], row["test_ret_moy_pct"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Edge full-sample vs edge sur p?riode de test")
            ax.set_xlabel("Rendement moyen du meilleur backtest complet (%)")
            ax.set_ylabel("Rendement moyen sur test chronologique (%)")
            generated.append(save_current_fig("03_full_period_vs_test_edge.png"))

    if master_view is not None and {"ticker", "random_side_percentile", "cost_break_even_bps"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["random_side_percentile", "cost_break_even_bps"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["random_side_percentile"], plot_df["cost_break_even_bps"], color="#f28e2b", alpha=0.8)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["random_side_percentile"], row["cost_break_even_bps"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Percentile random-side vs co?t break-even")
            ax.set_xlabel("Percentile du test random-side")
            ax.set_ylabel("Co?t break-even (bps)")
            generated.append(save_current_fig("04_random_side_vs_cost_break_even.png"))

    wf_ratio = None
    if master_view is not None and {"ticker", "walk_forward_positive_ratio"}.issubset(master_view.columns):
        wf_ratio = master_view[["ticker", "walk_forward_positive_ratio"]].dropna().copy()
    elif walk_forward is not None and "ticker" in walk_forward.columns:
        ret_col = next((c for c in ["ret_moy_pct", "ret_moy_%", "best_ret_moy_pct"] if c in walk_forward.columns), None)
        if ret_col:
            wf_ratio = (walk_forward.assign(_positive=walk_forward[ret_col] > 0)
                        .groupby("ticker", as_index=False)["_positive"].mean()
                        .rename(columns={"_positive": "walk_forward_positive_ratio"}))

    if wf_ratio is not None and not wf_ratio.empty:
        top_wf = wf_ratio.sort_values("walk_forward_positive_ratio", ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top_wf["ticker"].astype(str), top_wf["walk_forward_positive_ratio"], color="#79706e")
        ax.invert_yaxis()
        ax.set_title("Ratio positif en walk-forward par action")
        ax.set_xlabel("Part des p?riodes walk-forward positives")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("05_walk_forward_positive_ratio.png"))

    if not generated:
        display(Markdown("Aucun graphique de secours n'a pu ?tre g?n?r? parce que les CSV n?cessaires sont absents."))

## Interpr?tation

Cette cellule ?crit une interpr?tation courte ? partir du master summary charg?, pour que les exemples viennent des donn?es au lieu d'?tre ?crits en dur.

In [ ]:
def names_from(df, mask, limit=5):
    if df is None or "ticker" not in df.columns:
        return []
    sub = df.loc[mask].copy()
    if "confidence_score" in sub.columns:
        sub = sub.sort_values("confidence_score", ascending=False)
    return sub["ticker"].astype(str).head(limit).tolist()

if master_view is None:
    display(Markdown("L'interpr?tation se mettra ? jour quand `single_country_master_summary.csv` sera disponible."))
else:
    df = master_view.copy()
    cls = df["classification"].astype(str) if "classification" in df.columns else pd.Series([""] * len(df), index=df.index)
    a_names = names_from(df, cls.str.startswith("A"))
    rejected_names = names_from(df, cls.str.contains("D_REJECT|INSUFFICIENT", case=False, na=False))
    fragile_names = names_from(df, cls.str.startswith("B") | cls.str.startswith("C"))

    msg = f"Le premier batch US charg? contient {len(df)} noms. "
    if a_names:
        msg += "Les candidats A les plus propres sont " + ", ".join(a_names) + ". "
    else:
        msg += "Il n'y a pas de candidat A dans le r?sum? charg?. "
    if fragile_names:
        msg += "Un groupe plus large semble plus fragile ou plus d?pendant du VWAP, par exemple " + ", ".join(fragile_names) + ". "
    if rejected_names:
        msg += "Les noms rejet?s ou avec donn?es insuffisantes incluent " + ", ".join(rejected_names) + ". "
    msg += "Le point principal n'est pas seulement le rendement full-sample. Si la r?gle s?lectionn?e sur le train ?choue sur la p?riode de test, je ne veux pas consid?rer le r?sultat complet comme fiable."

    display(Markdown(msg))

## Pourquoi Des Noms Profitables Peuvent ?tre Rejet?s

Une action peut avoir un backtest full-sample positif et ?tre quand m?me rejet?e.

?a arrive quand le mod?le trouve une configuration profitable sur tout l'historique, mais que la configuration s?lectionn?e sur la premi?re partie ne marche pas sur la p?riode de test plus r?cente. Dans ce cas, le r?sultat peut ?tre sur-optimis?, d?pendant d'un r?gime pr?cis, ou surtout li? au filtre VWAP plut?t qu'au signal macro.

Le tableau ci-dessous cherche les actions o? :

```text
best_ret_moy_pct > 0
test_ret_moy_pct <= 0
```

In [ ]:
if master_view is None:
    display(Markdown("Impossible de construire ce tableau pour l'instant, car le master summary manque."))
elif {"best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
    rejected_profitable = master_view[
        (pd.to_numeric(master_view["best_ret_moy_pct"], errors="coerce") > 0)
        & (pd.to_numeric(master_view["test_ret_moy_pct"], errors="coerce") <= 0)
    ].copy()

    cols = ["ticker", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "classification", "main_warning"]
    cols = [c for c in cols if c in rejected_profitable.columns]

    if rejected_profitable.empty:
        display(Markdown("Aucune ligne charg?e ne correspond ? ce filtre."))
    else:
        display(rejected_profitable[cols].sort_values("best_ret_moy_pct", ascending=False))
else:
    display(Markdown("Les colonnes n?cessaires ne sont pas pr?sentes dans le master summary charg?."))

## Avertissement Principal

L'avertissement principal est que la configuration US actuelle contient beaucoup d'entr?es `PRE_OPEN_SAME_DAY`.

? cause de ?a, je ne dois pas pr?senter ce travail comme une r?action high-frequency pure aux publications macro. L'interpr?tation la plus honn?te est que je teste si les jours de publication macro cr?ent un environnement intraday o? une r?gle VWAP peut capter des patterns directionnels ou contrariants.

C'est quand m?me int?ressant, mais ce n'est pas la m?me affirmation. Avant de traiter un r?sultat comme une vraie id?e de trading, je voudrais des timestamps plus propres, plus de simulations al?atoires, et des hypoth?ses d'ex?cution plus conservatrices.

## Prochaines ?tapes

- Relancer les meilleurs candidats avec plus de simulations random.
- Nettoyer les noms de tickers et supprimer les fichiers bruts dupliqu?s avant le run final.
- Tester `limit_touch` plus en profondeur, parce que c'est plus proche d'une ex?cution r?aliste.
- Am?liorer la gestion des timestamps, surtout pour les publications US pr?-ouverture.
- Tester un univers plus large mais contr?l?.
- Construire une petite version paper-trading seulement apr?s validation plus stricte des candidats.